# InternHunt — Soft Computing Resume Classifier

**Upgrade: Logistic Regression → Neural Network (MLPClassifier)**

| Component | Old | New |
|-----------|-----|-----|
| Classifier | Logistic Regression (linear) | MLPClassifier — 128→64 hidden layers (non-linear) |
| TF-IDF features | 5000, max_df=0.95 | 6000, max_df=0.9, sublinear_tf=True |
| Output | Single prediction string | Top-3 predictions with fuzzy confidence labels |
| SC Concept | Hard computing / linear model | Neural Network + Fuzzy Logic |

**Fuzzy Labels:** probability > 0.7 → `High` | 0.4–0.7 → `Medium` | < 0.4 → `Low`

In [1]:
# 1) Setup
!pip -q install scikit-learn pandas joblib
!pip install numpy==2.4.4 pandas==3.0.2 scikit-learn==1.8.0


In [2]:
# 2) Upload the CSV (choose UpdatedResumeDataSet.csv in the dialog)
from google.colab import files
uploaded = files.upload()

Saving UpdatedResumeDataSet.csv to UpdatedResumeDataSet (4).csv


In [3]:
# 3) Pin scikit-learn to a stable version for reproducibility
!pip install --upgrade scikit-learn==1.7.2
import sklearn
print("✅ Using scikit-learn version:", sklearn.__version__)

  Using cached scikit_learn-1.7.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
Using cached scikit_learn-1.7.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.5 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.8.0
    Uninstalling scikit-learn-1.8.0:
      Successfully uninstalled scikit-learn-1.8.0
✅ Using scikit-learn version: 1.7.2


In [4]:
# ============================================================
# STEP 0: RUN THIS IN A NEW CELL AT THE TOP OF COLAB
# This ensures binary compatibility with the 2026 Streamlit environment
# !pip install numpy==2.4.4 pandas==3.0.2 scikit-learn==1.8.0
# ============================================================

# ============================================================
# CELL 4 — FINAL FIXED VERSION (Paste this in Colab)
# ============================================================

import os
import pandas as pd
import re
import joblib
import sklearn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

# ── Config ────────────────────────────────────────────────
CSV                   = "UpdatedResumeDataSet.csv"
TEXT_COL              = "Resume"
LABEL_COL             = "Category"
MIN_SAMPLES_PER_CLASS = 4
TEST_SIZE             = 0.2
RANDOM_STATE          = 42


# ── Fuzzy Label Helper ────────────────────────────────────
def fuzzy_label(probability: float) -> str:
    if probability > 0.70:
        return "High"
    elif probability >= 0.40:
        return "Medium"
    else:
        return "Low"


# ── Text Cleaning ─────────────────────────────────────────
def clean(t: str) -> str:
    t = str(t)
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


# ── Preprocessing ─────────────────────────────────────────
def preprocess_data(df, text_col, label_col, min_samples=MIN_SAMPLES_PER_CLASS):
    df[text_col] = df[text_col].astype(str).map(clean)
    class_counts = df[label_col].value_counts()
    valid_classes = class_counts[class_counts >= min_samples].index
    df_filtered = df[df[label_col].isin(valid_classes)].reset_index(drop=True)
    return df_filtered


# ── Pipeline — SIZE AND ACCURACY OPTIMISED ────────────────
def create_model_pipeline():
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=3000,      # Optimised for size
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.90,
            sublinear_tf=True,
            stop_words='english',
            lowercase=True
        )),
        ('clf', MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation='relu',
            solver='adam',
            alpha=1e-4,
            learning_rate_init=0.001,
            max_iter=1000,            # Higher iterations for better confidence
            tol=1e-4,
            random_state=RANDOM_STATE
        ))
    ])


# ── Training + Evaluation ─────────────────────────────────
def evaluate_model(pipeline, X_train, X_test, y_train, y_test):
    print("\nTraining MLP... (may take 1-2 mins)")
    pipeline.fit(X_train, y_train)
    clf_step = pipeline.named_steps['clf']
    print(f"Iterations: {clf_step.n_iter_} / {clf_step.max_iter}")

    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    return pipeline, accuracy


# ── Main ──────────────────────────────────────────────────
def main():
    try:
        df = pd.read_csv(CSV)[[TEXT_COL, LABEL_COL]].dropna()
        df_processed = preprocess_data(df, TEXT_COL, LABEL_COL)

        X_train, X_test, y_train, y_test = train_test_split(
            df_processed[TEXT_COL], df_processed[LABEL_COL],
            test_size=TEST_SIZE, random_state=RANDOM_STATE
        )

        pipeline = create_model_pipeline()
        trained_pipeline, final_accuracy = evaluate_model(pipeline, X_train, X_test, y_train, y_test)

        # Save with max compression
        model_filename = "resume_classifier_v2.pkl"
        metadata = {
            "model": trained_pipeline,
            "sklearn_version": sklearn.__version__,
            "model_type": "MLPClassifier",
            "architecture": "TF-IDF(3000) -> MLP(128,64)",
            "fuzzy_labels": {"High": ">0.70", "Medium": "0.40-0.70", "Low": "<0.40"}
        }

        joblib.dump(metadata, model_filename, compress=('zlib', 9))
        size_mb = os.path.getsize(model_filename) / (1024 * 1024)
        print(f"\n✅ Saved: {model_filename} ({size_mb:.2f} MB)")

        return trained_pipeline
    except Exception as e:
        print(f"Error: {e}")

model = main()


Training MLP... (may take 1-2 mins)
Iterations: 84 / 1000
Test Accuracy: 0.9948 (99.48%)

✅ Saved: resume_classifier_v2.pkl (11.56 MB)


In [7]:
# ============================================================
# 6) Download the upgraded model
# ============================================================
from google.colab import files
files.download("resume_classifier_v2.pkl")
print("✅ Download started — replace the existing resume_classifier_v2.pkl in your project root.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started — replace the existing resume_classifier_v2.pkl in your project root.


# 📊 Performance Metrics

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("UpdatedResumeDataSet.csv")[["Resume", "Category"]]
X_train, X_test, y_train, y_test = train_test_split(
    df["Resume"], df["Category"], test_size=0.2, stratify=df["Category"], random_state=42
)

In [9]:
y_pred = model.predict(X_test)

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print("Model Type : Neural Network (MLPClassifier, 128→64 hidden layers)")
print("-" * 55)
print("Accuracy  :", round(accuracy, 3))
print("Precision :", round(precision, 3))
print("Recall    :", round(recall, 3))
print("F1 Score  :", round(f1, 3))

Model Type : Neural Network (MLPClassifier, 128→64 hidden layers)
-------------------------------------------------------
Accuracy  : 1.0
Precision : 1.0
Recall    : 1.0
F1 Score  : 1.0


In [11]:
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Classification Report:
                            precision    recall  f1-score   support

                 Advocate       1.00      1.00      1.00         4
                     Arts       1.00      1.00      1.00         7
       Automation Testing       1.00      1.00      1.00         5
               Blockchain       1.00      1.00      1.00         8
         Business Analyst       1.00      1.00      1.00         6
           Civil Engineer       1.00      1.00      1.00         5
             Data Science       1.00      1.00      1.00         8
                 Database       1.00      1.00      1.00         7
          DevOps Engineer       1.00      1.00      1.00        11
         DotNet Developer       1.00      1.00      1.00         5
            ETL Developer       1.00      1.00      1.00         8
   Electrical Engineering       1.00      1.00      1.00         6
                       HR       1.00      1.00      1.00         9
                   Hadoop       1.00